# Full ML Pipeline: Hotel Booking Cancellation Prediction

## Context

Imagine you work at a hotel chain. Every day, guests make reservations, but a significant portion cancel before arrival. If the hotel could **predict at booking time** which reservations are likely to be canceled, it could take action: overbooking strategically, requiring deposits from high-risk bookings, or sending retention offers.

Our task is to build a model that, given the features of a reservation (lead time, room type, country, deposit type, etc.), predicts whether the guest will cancel. This is a **binary classification** problem: for each booking, the answer is either "canceled" (1) or "not canceled" (0).

Throughout this notebook, you will **reason about every decision** in the pipeline. Why this encoding? Why this validation strategy? Why this metric? The goal is to build the engineering intuition you'll need for your Kaggle competition project.


**Dataset:** Synthetic hotel booking data (~12k rows, 22 features).
**Target:** `is_canceled` (binary: will the guest cancel?)

### 📏 How we measure progress

Throughout this notebook, a **performance tracker** records the ROC-AUC score after each pipeline step. This lets you see exactly how much each decision improves (or hurts!) the model.

All scores use **5-fold Stratified K-Fold cross-validation on the training set only**. The same model (`GradientBoostingClassifier` with 100 trees, depth 4) is used as a consistent "ruler" in the early steps, so any improvement comes from better data, not a better algorithm.

Let's go!🚀

In [ ]:
#@title 🗺️ Notebook Roadmap { display-mode: "form" }
from IPython.display import HTML, display

display(HTML('''
<style>
.rm{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);
    border-radius:18px;padding:22px 20px;margin:8px 0;border:1px solid #ecebff}
.rm-h{font-size:20px;font-weight:800;color:#3b2d6b;margin:0 0 4px}
.rm-s{font-size:12px;color:#6b6685;margin:0 0 16px}
.rm-list{display:flex;flex-direction:column;gap:0}
.rm-item{display:flex;align-items:flex-start;gap:12px;padding:8px 6px;border-radius:10px}
.rm-item:hover{background:#ffffffaa}
.rm-num{flex:0 0 34px;height:34px;border-radius:50%;display:flex;align-items:center;justify-content:center;
        font-size:15px;font-weight:800;color:#fff;box-shadow:0 4px 10px rgba(102,126,234,.35)}
.rm-body{padding-top:2px}
.rm-t{font-weight:800;font-size:13.5px;color:#2c2350}
.rm-d{font-size:11.5px;color:#8b86a6;margin-top:2px;line-height:1.35}
.rm-line{width:2px;flex:0 0 2px;background:#e3ddfb;margin-left:17px}
</style>
<div class="rm">
  <div class="rm-h">🗺️ The Pipeline, Step by Step</div>
  <div class="rm-s">Eleven stops from raw bookings to a Kaggle submission &mdash; each one builds on the last.</div>
  <div class="rm-list">
    <div class="rm-item"><div class="rm-num" style="background:linear-gradient(135deg,#667eea,#667eeacc)">0</div>
      <div class="rm-body"><div class="rm-t">🧠 Understand the Problem</div><div class="rm-d">Interactive quiz + spot the data leak in a raw baseline</div></div></div>
    <div class="rm-item"><div class="rm-num" style="background:linear-gradient(135deg,#7375e6,#7375e6cc)">1</div>
      <div class="rm-body"><div class="rm-t">🔍 EDA</div><div class="rm-d">Visualize &rarr; decide &rarr; preprocess</div></div></div>
    <div class="rm-item"><div class="rm-num" style="background:linear-gradient(135deg,#8177e0,#8177e0cc)">2</div>
      <div class="rm-body"><div class="rm-t">🧹 Preprocessing</div><div class="rm-d">Multiple-choice questions before each cleaning step</div></div></div>
    <div class="rm-item"><div class="rm-num" style="background:linear-gradient(135deg,#8f6fda,#8f6fdacc)">3</div>
      <div class="rm-body"><div class="rm-t">🛠️ Feature Engineering</div><div class="rm-d">Datetime, aggregation &amp; interaction features</div></div></div>
    <div class="rm-item"><div class="rm-num" style="background:linear-gradient(135deg,#9d6ed4,#9d6ed4cc)">4</div>
      <div class="rm-body"><div class="rm-t">🎯 Feature Selection</div><div class="rm-d">Correlation &amp; Random Forest importance</div></div></div>
    <div class="rm-item"><div class="rm-num" style="background:linear-gradient(135deg,#ab6ecf,#ab6ecfcc)">5</div>
      <div class="rm-body"><div class="rm-t">✅ Validation Strategy</div><div class="rm-d">Making sure our scores can be trusted</div></div></div>
    <div class="rm-item"><div class="rm-num" style="background:linear-gradient(135deg,#b96dc9,#b96dc9cc)">6</div>
      <div class="rm-body"><div class="rm-t">📐 Baseline Model</div><div class="rm-d">A simple reference point: is this problem learnable?</div></div></div>
    <div class="rm-item"><div class="rm-num" style="background:linear-gradient(135deg,#c76dc4,#c76dc4cc)">7</div>
      <div class="rm-body"><div class="rm-t">⚔️ Compare Models</div><div class="rm-d">From simple models to gradient-boosted ensembles</div></div></div>
    <div class="rm-item"><div class="rm-num" style="background:linear-gradient(135deg,#d56cbe,#d56cbecc)">8</div>
      <div class="rm-body"><div class="rm-t">⚙️ Hyperparameter Optimization</div><div class="rm-d">Grid Search &amp; Optuna (pre-computed)</div></div></div>
    <div class="rm-item"><div class="rm-num" style="background:linear-gradient(135deg,#e36cb9,#e36cb9cc)">9</div>
      <div class="rm-body"><div class="rm-t">🧩 Ensemble Methods</div><div class="rm-d">Averaging, weighted blends, stacking &amp; bonus tricks</div></div></div>
    <div class="rm-item"><div class="rm-num" style="background:linear-gradient(135deg,#f16bb3,#f16bb3cc)">10</div>
      <div class="rm-body"><div class="rm-t">📊 Error Analysis &amp; 📤 Kaggle Submission</div><div class="rm-d">Diagnose mistakes, then generate solution.csv</div></div></div>
  </div>
</div>'''))


In [ ]:
# ============================================================
# @title SETUP: run this cell first
# ============================================================
import warnings; warnings.filterwarnings('ignore')
import subprocess, sys
for pkg in ['xgboost', 'lightgbm', 'catboost', 'optuna']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import HTML, display

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 4)
pd.set_option('display.float_format', lambda x: f'{x:.3f}')
pd.set_option('display.max_columns', 30)

# Performance tracker
performance_log = []
def log_score(step, description, score, metric='ROC-AUC'):
    '''Log a score and display the running tracker.'''
    performance_log.append({'Step': step, 'Description': description,
                            'Metric': metric, 'Score': round(score, 4)})
    display(pd.DataFrame(performance_log).style
            .background_gradient(subset=['Score'], cmap='RdYlGn')
            .format({'Score': '{:.4f}'}))

In [ ]:
#@title Load data from GitHub
!git clone -b main https://github.com/eth-fdd-fs26/FDD-WE1-public.git 2>/dev/null || echo "Already cloned"

train = pd.read_csv('FDD-WE1-public/data/booking_train.csv', parse_dates=['booking_date', 'arrival_date'])
test  = pd.read_csv('FDD-WE1-public/data/booking_test.csv',  parse_dates=['booking_date', 'arrival_date'])
print(f"Train: {train.shape},  Test: {test.shape}")

## 0. 🧠 Understand the Problem

In [ ]:
#@title Q1: Understanding the problem
quiz_html = '''
<div style="border:2px solid #4A90D9; border-radius:12px; padding:20px; margin:10px 0; background:#f0f7ff;">
<h4>🧠 Before we start: answer these three questions:</h4>

<p><b>Q1: What type of problem is this?</b></p>
<input type="radio" name="q1" id="q1a"><label for="q1a"> A) Regression (predict a continuous value)</label><br>
<input type="radio" name="q1" id="q1b"><label for="q1b"> B) Binary classification (predict 0 or 1)</label><br>
<input type="radio" name="q1" id="q1c"><label for="q1c"> C) Multi-class classification (predict 1 of N classes)</label><br>
<input type="radio" name="q1" id="q1d"><label for="q1d"> D) Clustering (group similar items)</label><br>

<div style="background:#eef6ff; border-left:4px solid #4A90D9; padding:12px 16px; margin:12px 0; border-radius:6px;">
<p><b>📖 Why ROC-AUC?</b></p>
<p>In a Kaggle competition, the evaluation metric is set beforehand: for us it's ROC-AUC. But why is this a good choice?</p>
<p>ROC-AUC measures how well the model ranks positive cases (cancellations) above negative ones, across all possible classification thresholds. A perfect model gets AUC = 1.0; a random guess gets AUC = 0.5.</p>
<p>Unlike accuracy, AUC is not fooled by class imbalance. If 63% of bookings are NOT canceled, a model that always predicts "not canceled" gets 63% accuracy (looks decent!) but AUC = 0.5 (exposed as useless). AUC reveals whether the model has actually learned something.</p>
</div>

<p><b>Q2: Given the explanation above, why is ROC-AUC particularly appropriate for this problem, compared to plain accuracy?</b></p>
<input type="radio" name="q2" id="q2a"><label for="q2a"> A) Because it's faster to compute than accuracy</label><br>
<input type="radio" name="q2" id="q2b"><label for="q2b"> B) Because it isn't fooled by class imbalance: it measures ranking quality across all thresholds</label><br>
<input type="radio" name="q2" id="q2c"><label for="q2c"> C) Because it only works when classes are perfectly balanced</label><br>
<input type="radio" name="q2" id="q2d"><label for="q2d"> D) Because it doesn't require predicted probabilities</label><br>

<p><b>Q3: A model always predicts "not canceled." With 63% non-canceled bookings, what scores does it get?</b></p>
<input type="radio" name="q3" id="q3a"><label for="q3a"> A) Accuracy = 63%, AUC = 0.63</label><br>
<input type="radio" name="q3" id="q3b"><label for="q3b"> B) Accuracy = 63%, AUC = 0.50</label><br>
<input type="radio" name="q3" id="q3c"><label for="q3c"> C) Accuracy = 50%, AUC = 0.50</label><br>
<input type="radio" name="q3" id="q3d"><label for="q3d"> D) Accuracy = 37%, AUC = 0.37</label><br>

<br>
<details><summary style="cursor:pointer; background:#4A90D9; color:white; padding:8px 16px; border-radius:6px; display:inline-block;"><b>📝 Submit & Check Answers</b></summary>
<div style="margin-top:12px; padding:12px; background:#e8f5e9; border-radius:8px;">
<p>✅ <b>Q1: B) Binary classification</b>: the target is 0 (not canceled) or 1 (canceled).</p>
<p>✅ <b>Q2: B)</b> ROC-AUC is not fooled by class imbalance: it measures ranking quality regardless of how skewed the classes are. A model predicting the same thing for everyone gets AUC = 0.5 (exposed as useless), while accuracy can be inflated by the majority class.</p>
<p>✅ <b>Q3: B) Accuracy = 63%, AUC = 0.50</b>: the model gets 63% accuracy by always predicting the majority class, but AUC = 0.5 because it has zero discriminative ability (it ranks everyone the same).</p>
</div>
</details>
</div>
'''
display(HTML(quiz_html))

In [ ]:
#@title Verify with the data
print("Target distribution:")
print(train['is_canceled'].value_counts())
print(f"\nCancellation rate: {train['is_canceled'].mean():.1%}")
print(f"Naive baseline accuracy: {1 - train['is_canceled'].mean():.1%}")
print(f"Naive baseline AUC: 0.500")

In [ ]:
#@title 👀 Peek at the Raw Data
train.tail(5)

### 📋 Column Definitions

- **`lead_time`**: days between booking and arrival (0 = same-day, 300 = ~10 months ahead)
- **`avg_daily_rate`**: average price per night in euros
- **`agent_id`**: booking intermediary (travel agency, Booking.com, etc.)
- **`cancellation_fee_charged`**: whether a fee was charged, but *when* does this happen?

In [ ]:
#@title Q2: Spot the leak
quiz_html = '''
<div style="border:2px solid #E8593C; border-radius:12px; padding:20px; margin:10px 0; background:#fff5f5;">
<h4>🔍 Which column reveals information from the FUTURE?</h4>
<p>We want to predict cancellation <b>at booking time</b>. One column contains info the hotel would only know AFTER the guest cancels.</p>
<input type="radio" name="ql" id="qla"><label for="qla"> A) lead_time</label><br>
<input type="radio" name="ql" id="qlb"><label for="qlb"> B) avg_daily_rate</label><br>
<input type="radio" name="ql" id="qlc"><label for="qlc"> C) cancellation_fee_charged</label><br>
<input type="radio" name="ql" id="qld"><label for="qld"> D) agent_id</label><br>
<br>
<details><summary style="cursor:pointer; background:#E8593C; color:white; padding:8px 16px; border-radius:6px; display:inline-block;"><b>📝 Check Answer</b></summary>
<div style="margin-top:12px; padding:12px; background:#fce4ec; border-radius:8px;">
✅ <b>C) cancellation_fee_charged</b>: the hotel charges this fee AFTER the cancellation happens. Including it lets the model "cheat" by looking at the answer. This is called a <b>data leak</b>. Let us verify below…
</div></details>
</div>
'''
display(HTML(quiz_html))

In [ ]:
#@title 📏 Raw Baseline: The Leak in Action
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier

raw = train.drop(columns=['booking_id', 'booking_date', 'arrival_date'])
raw_num = raw.select_dtypes(include='number').copy().fillna(-999)
y_raw = raw_num.pop('is_canceled')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
ruler = GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42)

s_leak = cross_val_score(ruler, raw_num, y_raw, cv=cv, scoring='roc_auc')
s_clean = cross_val_score(ruler, raw_num.drop(columns=['cancellation_fee_charged']), y_raw, cv=cv, scoring='roc_auc')

print(f"WITH leak:    AUC = {s_leak.mean():.4f} ± {s_leak.std():.4f}  ← suspiciously high!")
print(f"WITHOUT leak: AUC = {s_clean.mean():.4f} ± {s_clean.std():.4f}  ← true starting point")
log_score('0', 'Raw baseline (numeric only, no leak)', s_clean.mean())

## 1. 🔍 Exploratory Data Analysis (EDA)

**Goal:** Understand the data well enough to make preprocessing decisions. Every plot should answer: *"What should I DO based on what I SEE?"*

### Feature types in this dataset

- **Numerical, continuous:** `avg_daily_rate` (price)
- **Numerical, discrete:** `lead_time`, `stays_weekend`, `stays_weekday`, `adults`, `children`, `babies`, `special_requests`, `previous_cancellations` (counts of things)
- **Categorical, nominal (no order):** `meal_plan`, `country`, `market_segment`, `deposit_type`
- **Categorical, ordinal (ordered):** `room_type` (Standard < Deluxe < Suite)
- **Datetime:** `booking_date`, `arrival_date`, raw dates we'll turn into month, day of week, and other engineered features
- **Binary:** `is_repeated_guest`, already numeric (0/1)

### Why visualize?

- **Histograms** → spot skewness → decide on log transforms
- **Box plots** → spot outliers + see if a feature separates the classes
- **Correlation heatmap** → find redundancy + detect data leaks
- **Missing value plots** → determine imputation strategy (see below)
- **Categorical bar plots** → see which categories behave differently → inform encoding

### ❓ Missing Value Types: Why This Matters for Imputation

Not all missing data is the same. The *reason* it's missing determines how you should fill it

For example, `previous_cancellations` is missing mostly for first-time guests. Imputing with the mean (~0.7 cancellations) would wrongly suggest first-time guests have a cancellation history.

The right approach: impute with 0 (the logical value) and create a binary flag `prev_cancel_missing` to preserve the signal that this guest is new.

For `children`, the mean is ~0.4, but you can't have 0.4 children. The mode (0) is more logical.

In [ ]:
#@title 📉 1.1 Missing Values Overview
missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 3))
missing.plot.barh(ax=ax, color='#4A90D9'); ax.set_title('Missing values'); ax.set_xlabel('Count')
plt.tight_layout(); plt.show()
print(f"\n% missing:\n{(missing / len(train) * 100).round(1)}")

In [ ]:
#@title 🔍 1.2 Who Is Missing?
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
mg = train.groupby('is_repeated_guest')['previous_cancellations'].apply(lambda x: x.isnull().mean())
axes[0].bar(['First-time (0)', 'Repeated (1)'], mg.values, color=['#E8593C', '#4A90D9'])
axes[0].set_title('previous_cancellations: missing rate by guest type'); axes[0].set_ylabel('% missing')
mt = train.groupby('is_canceled')['previous_cancellations'].apply(lambda x: x.isnull().mean())
axes[1].bar(['Not canceled (0)', 'Canceled (1)'], mt.values, color=['#5DCAA5', '#E8593C'])
axes[1].set_title('previous_cancellations: missing rate by target'); axes[1].set_ylabel('% missing')
plt.tight_layout(); plt.show()

In [ ]:
#@title Q3: Interpreting the Missing Value Plots
quiz_html = '''
<div style="border:2px solid #4A90D9; border-radius:12px; padding:20px; margin:10px 0; background:#f0f7ff;">
<h4>🔍 Look at the two plots above and answer:</h4>

<p><b>Q1: The left plot shows that <code>previous_cancellations</code> is missing for ~20%% of first-time guests but 0%% of repeated guests. What does this tell you about <i>why</i> the data is missing?</b></p>
<input type="radio" name="mq1" id="mq1a"><label for="mq1a"> A) The missing values are random and unrelated to any other column</label><br>
<input type="radio" name="mq1" id="mq1b"><label for="mq1b"> B) The data is missing <i>because</i> the guest is new: the hotel has no cancellation history to record for first-time guests</label><br>
<input type="radio" name="mq1" id="mq1c"><label for="mq1c"> C) There is a bug in the data collection pipeline that drops values at random</label><br>
<input type="radio" name="mq1" id="mq1d"><label for="mq1d"> D) Repeated guests always cancel, so their data is always present</label><br>

<p><b>Q2: The right plot shows that the missing rate is roughly similar for both canceled (~17.5%%) and not-canceled (~16.5%%) bookings. What does this suggest about using a simple overall mean to fill in the missing values?</b></p>
<input type="radio" name="mq2" id="mq2a"><label for="mq2a"> A) The mean is fine because missingness doesn\'t depend on the target</label><br>
<input type="radio" name="mq2" id="mq2b"><label for="mq2b"> B) The mean would be misleading: we already know from the left plot that missingness is driven by guest type, so imputing with the overall mean (~0.7 cancellations) would wrongly give first-time guests a cancellation history they don\'t have</label><br>
<input type="radio" name="mq2" id="mq2c"><label for="mq2c"> C) We should drop the column entirely since both classes have similar missing rates</label><br>
<input type="radio" name="mq2" id="mq2d"><label for="mq2d"> D) The right plot proves the data is missing completely at random</label><br>

<br>
<details><summary style="cursor:pointer; background:#4A90D9; color:white; padding:8px 16px; border-radius:6px; display:inline-block;"><b>📝 Check Answers</b></summary>
<div style="margin-top:12px; padding:12px; background:#e8f5e9; border-radius:8px;">
<p>✅ <b>Q1: B)</b> The pattern is clear: missingness is tied to guest type. The hotel has no prior record for someone who has never booked before, so the field is left blank. This is <i>not</i> random.</p>
<p>✅ <b>Q2: B)</b> Even though the right plot shows similar missing rates across the target, the left plot already revealed the real driver. Imputing with the overall mean would assign ~0.7 past cancellations to brand-new guests who logically have 0. The correct approach: impute with 0 and add a binary flag (<code>prev_cancel_missing</code>) to preserve the "new guest" signal.</p>
</div>
</details>
</div>
'''
display(HTML(quiz_html))

### 📈 Skewness and log transforms

**Skewness** measures how asymmetric a distribution is. A skewness near 0 means roughly symmetric; a large positive skewness means a long right tail (a few very large values pulling the mean above the median). `avg_daily_rate` has a few very expensive bookings that stretch the distribution far to the right.

Why does this matter? Linear models (Logistic Regression, SVM) fit a straight-line relationship, and a long tail lets a handful of extreme rows dominate that fit. A **log transform** (`log1p`, i.e. `log(1+x)`, safe for zeros) compresses large values much more than small ones, pulling the tail in and making the distribution closer to symmetric. Tree-based models (Random Forest, XGBoost, LightGBM) split on thresholds and don't care about the scale or shape of a feature, so log transforms mostly matter for linear models here.

**Rule of thumb used below:** |skewness| > 1 is a reasonable trigger to consider a log transform.

In each histogram below, the **red dashed line marks the median** of that feature. When the red line sits far from the center of the bulk of the bars, or the tail stretches out much further on one side than the line, that's a visual cue of skew.

In [ ]:
#@title 1.3 Distributions: spot skewness
numerical_cols = ['lead_time', 'avg_daily_rate', 'stays_weekday', 'stays_weekend',
                  'adults', 'special_requests', 'previous_bookings_not_canceled']
fig, axes = plt.subplots(2, 4, figsize=(16, 8)); axes = axes.flatten()
for i, col in enumerate(numerical_cols):
    train[col].hist(bins=40, ax=axes[i], color='#4A90D9', edgecolor='white')
    axes[i].set_title(f'{col}\nskewness = {train[col].skew():.2f}')
    axes[i].axvline(train[col].median(), color='red', linestyle='--')
axes[-1].axis('off')
plt.suptitle('Distributions: |skewness| > 1 may need log transform for linear models', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
#@title Q4: Which features need log transform?
quiz_html = '''
<div style="border:2px solid #4A90D9; border-radius:12px; padding:20px; margin:10px 0; background:#f0f7ff;">
<h4>🧠 Based on the distributions above:</h4>

<p><b>Which features have |skewness| > 1 and should be log-transformed for linear models?</b></p>
<input type="checkbox" id="sk1"><label for="sk1"> lead_time </label><br>
<input type="checkbox" id="sk2"><label for="sk2"> avg_daily_rate </label><br>
<input type="checkbox" id="sk3"><label for="sk3"> stays_weekday </label><br>
<input type="checkbox" id="sk4"><label for="sk4"> adults </label><br>
<input type="checkbox" id="sk5"><label for="sk5"> special_requests </label><br>
<br>
<details><summary style="cursor:pointer; background:#4A90D9; color:white; padding:8px 16px; border-radius:6px; display:inline-block;"><b>📝 Check Answers</b></summary>
<div style="margin-top:12px; padding:12px; background:#e8f5e9; border-radius:8px;">
✅ <b>lead_time (1.80)</b> and <b>avg_daily_rate (6.40)</b>: both have |skewness| > 1.<br>
❌ stays_weekday (0.43), adults (0.66), special_requests (0.67): all below 1, acceptable.<br><br>
<b>Note:</b> Log transform helps <i>linear</i> models (LogReg, SVM). Tree-based models (XGBoost, LightGBM) don't need it; they split on thresholds regardless of distribution shape. We'll apply <code>log1p</code> in the Feature Engineering section.
</div></details>
</div>
'''
display(HTML(quiz_html))

In [ ]:
#@title 1.4 Box plots: outliers + discriminative power
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
train.boxplot(column='avg_daily_rate', ax=axes[0]); axes[0].set_title('avg_daily_rate: outliers?')
train.boxplot(column='lead_time', by='is_canceled', ax=axes[1]); axes[1].set_title('lead_time by cancellation')
train.boxplot(column='avg_daily_rate', by='is_canceled', ax=axes[2]); axes[2].set_title('avg_daily_rate by cancellation')
plt.suptitle(''); plt.tight_layout(); plt.show()

In [ ]:
#@title Q5: Reading the box plots
quiz_html = '''
<div style="border:2px solid #4A90D9; border-radius:12px; padding:20px; margin:10px 0; background:#f0f7ff;">
<h4>🧠 Based on the box plots above:</h4>

<p><b>Q1: Most avg_daily_rate values sit under ~€200 (the box), but individual points scatter all the way up to ~€1500. Does that mean those high points are data errors?</b></p>
<input type="radio" name="bq1" id="bq1a"><label for="bq1a"> A) Yes, always remove rows like this</label><br>
<input type="radio" name="bq1" id="bq1b"><label for="bq1b"> B) Not necessarily: could be legitimate luxury bookings, worth capping (clipping) rather than deleting</label><br>
<input type="radio" name="bq1" id="bq1c"><label for="bq1c"> C) Yes, always replace them with the mean</label><br>

<p><b>Q2: In "lead_time by cancellation," canceled bookings show a visibly higher median lead_time (~75 days) than non-canceled ones (~48 days), plus a wider spread. What does that suggest?</b></p>
<input type="radio" name="bq2" id="bq2a"><label for="bq2a"> A) Bookings made further in advance are more likely to be canceled: plans made early have more time to fall through</label><br>
<input type="radio" name="bq2" id="bq2b"><label for="bq2b"> B) Lead time has no relationship with cancellation</label><br>
<input type="radio" name="bq2" id="bq2c"><label for="bq2c"> C) Canceled bookings are always made last-minute</label><br>

<p><b>Q3: In "avg_daily_rate by cancellation," the two boxes are almost fully overlapping. What does that tell us about avg_daily_rate as a standalone predictor?</b></p>
<input type="radio" name="bq3" id="bq3a"><label for="bq3a"> A) It's a very strong predictor on its own</label><br>
<input type="radio" name="bq3" id="bq3b"><label for="bq3b"> B) On its own it has weak discriminative power for this target, though it may still help combined with other features</label><br>
<input type="radio" name="bq3" id="bq3c"><label for="bq3c"> C) Price has no effect on hotel bookings at all</label><br>

<br>
<details><summary style="cursor:pointer; background:#4A90D9; color:white; padding:8px 16px; border-radius:6px; display:inline-block;"><b>📝 Check Answers</b></summary>
<div style="margin-top:12px; padding:12px; background:#e8f5e9; border-radius:8px;">
✅ <b>Q1: B)</b> A wide, sparse tail of high values isn't automatically an error: a €1,200 booking can be a real luxury suite. Clipping at a high percentile keeps the signal ("this was an expensive booking") without letting a handful of extreme rows dominate training.<br><br>
✅ <b>Q2: A)</b> The visual gap between the two medians is exactly the kind of signal a model can use: this matches what we already saw in the correlation heatmap (lead_time correlates positively with is_canceled).<br><br>
✅ <b>Q3: B)</b> Heavy visual overlap between classes means weak univariate signal. That doesn't mean the feature is useless: it can still contribute in combination with other features (we'll let Random Forest importance judge that later, in Feature Selection).
</div></details>
</div>
'''
display(HTML(quiz_html))

In [ ]:
#@title 1.5 Correlation heatmap
numeric_train = train.select_dtypes(include='number')
corr = numeric_train.corr()
fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax,
            vmin=-1, vmax=1, linewidths=0.5, square=True, annot_kws={"size": 7})
ax.set_title('Correlation Matrix'); plt.tight_layout(); plt.show()

In [ ]:
#@title Q6: Correlation insights
quiz_html = '''
<div style="border:2px solid #4A90D9; border-radius:12px; padding:20px; margin:10px 0; background:#f0f7ff;">
<h4>🧠 Based on the correlation heatmap:</h4>

<p><b>Q1: Which column has suspiciously high correlation (~0.90) with is_canceled?</b></p>
<input type="radio" name="qc1" id="qc1a"><label for="qc1a"> A) lead_time</label><br>
<input type="radio" name="qc1" id="qc1b"><label for="qc1b"> B) special_requests</label><br>
<input type="radio" name="qc1" id="qc1c"><label for="qc1c"> C) cancellation_fee_charged</label><br>
<input type="radio" name="qc1" id="qc1d"><label for="qc1d"> D) agent_id</label><br>

<p><b>Q2: What does a NEGATIVE correlation (e.g., special_requests vs is_canceled) mean?</b></p>
<input type="radio" name="qc2" id="qc2a"><label for="qc2a"> A) The feature is useless</label><br>
<input type="radio" name="qc2" id="qc2b"><label for="qc2b"> B) As one goes up, the other goes down</label><br>
<input type="radio" name="qc2" id="qc2c"><label for="qc2c"> C) There's an error in the data</label><br>
<br>
<details><summary style="cursor:pointer; background:#4A90D9; color:white; padding:8px 16px; border-radius:6px; display:inline-block;"><b>📝 Check Answers</b></summary>
<div style="margin-top:12px; padding:12px; background:#e8f5e9; border-radius:8px;">
✅ <b>Q1: C)</b> cancellation_fee_charged: this is the data leak we already identified.<br>
✅ <b>Q2: B)</b> Negative correlation means they move in opposite directions. More special requests → fewer cancellations (guests who make requests are more committed). Negative ≠ useless; it's equally predictive!
</div></details>

<p><b>Q3: Besides the leak, which feature shows the strongest legitimate (non-leak) correlation with is_canceled?</b></p>
<input type="radio" name="qc3" id="qc3a"><label for="qc3a"> A) lead_time (0.19)</label><br>
<input type="radio" name="qc3" id="qc3b"><label for="qc3b"> B) children (-0.08)</label><br>
<input type="radio" name="qc3" id="qc3c"><label for="qc3c"> C) parking_spaces (0.01)</label><br>
<input type="radio" name="qc3" id="qc3d"><label for="qc3d"> D) agent_id (0.03)</label><br>

<p><b>Q4: previous_cancellations (0.14, positive), special_requests (-0.13, negative), and is_repeated_guest (-0.11, negative) all have similar-sized correlations. What's a plausible story connecting the three?</b></p>
<input type="radio" name="qc4" id="qc4a"><label for="qc4a"> A) Guests who make more requests or have stayed before tend to be more invested/reliable, while a history of past cancellations predicts future ones</label><br>
<input type="radio" name="qc4" id="qc4b"><label for="qc4b"> B) These three columns are all measuring the same underlying thing</label><br>
<input type="radio" name="qc4" id="qc4c"><label for="qc4c"> C) The signs are just noise and don't mean anything</label><br>

<br>
<details><summary style="cursor:pointer; background:#4A90D9; color:white; padding:8px 16px; border-radius:6px; display:inline-block;"><b>📝 Check Q3 & Q4</b></summary>
<div style="margin-top:12px; padding:12px; background:#e8f5e9; border-radius:8px;">
✅ <b>Q3: A)</b> lead_time at 0.19 is the strongest legitimate signal, and it matches what we already saw in the box plots: canceled bookings tend to have a higher lead time.<br><br>
✅ <b>Q4: A)</b> It's a reasonable read of the data (correlation alone can't prove causation, but the story is consistent): guests who make special requests or who are repeat customers appear more committed and cancel less, while guests with a track record of cancellations are more likely to cancel again. This is exactly the kind of pattern we'll try to capture with modeling later.
</div></details>
</div>
'''
display(HTML(quiz_html))

In [ ]:
#@title 1.6 Categorical bar plots
cat_cols = ['meal_plan', 'country', 'market_segment', 'room_type', 'deposit_type']
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, col in enumerate(cat_cols):
    if col == 'country':
        top10 = train[col].value_counts().head(10).index
        cr = train[train[col].isin(top10)].groupby(col)['is_canceled'].mean().sort_values(ascending=False)
    else:
        cr = train.groupby(col)['is_canceled'].mean().sort_values(ascending=False)
    cr.plot.bar(ax=axes[i], color='#4A90D9', edgecolor='white')
    axes[i].set_title(col); axes[i].set_ylabel('Cancel rate')
    axes[i].axhline(y=train['is_canceled'].mean(), color='red', linestyle='--', alpha=0.7)
    axes[i].tick_params(axis='x', rotation=45)
plt.suptitle('Cancellation rate by category (red = overall average)', y=1.02)
plt.tight_layout(); plt.show()

### 🏷️ Encoding categorical variables

| Situation | Strategy |
|-----------|----------|
| Ordinal (natural order) | Label Encoding: 0, 1, 2 |
| Nominal, few values (<10) | One-Hot Encoding | meal_plan (4 values) |
| Nominal, many values (10+) | Target Encoding with smoothing | country (54 values) |

In [ ]:
#@title Q7: Choosing an encoding strategy
quiz_html = '''
<div style="border:2px solid #4A90D9; border-radius:12px; padding:20px; margin:10px 0; background:#f0f7ff;">
<h4>🧠 Based on the categorical bar plots above, how should we encode each column?</h4>

<p><b>Q1: room_type (Standard, Deluxe, Suite)</b></p>
<input type="radio" name="eq1" id="eq1a"><label for="eq1a"> A) One-Hot Encoding</label><br>
<input type="radio" name="eq1" id="eq1b"><label for="eq1b"> B) Label Encoding (it has a natural order)</label><br>
<input type="radio" name="eq1" id="eq1c"><label for="eq1c"> C) Target Encoding</label><br>

<p><b>Q2: meal_plan (4 values, no natural order)</b></p>
<input type="radio" name="eq2" id="eq2a"><label for="eq2a"> A) One-Hot Encoding</label><br>
<input type="radio" name="eq2" id="eq2b"><label for="eq2b"> B) Label Encoding: 0, 1, 2, 3</label><br>

<p><b>Q3: country (54 values)</b></p>
<input type="radio" name="eq3" id="eq3a"><label for="eq3a"> A) One-Hot Encoding (54 new columns)</label><br>
<input type="radio" name="eq3" id="eq3b"><label for="eq3b"> B) Target Encoding with smoothing</label><br>
<input type="radio" name="eq3" id="eq3c"><label for="eq3c"> C) Label Encoding: 0 to 53</label><br>

<p><b>Q4: Why "with smoothing" for target encoding, instead of just the raw per-category mean of the target?</b></p>
<input type="radio" name="eq4" id="eq4a"><label for="eq4a"> A) To avoid overfitting: rare categories with very few samples get their mean blended toward the global mean</label><br>
<input type="radio" name="eq4" id="eq4b"><label for="eq4b"> B) Smoothing makes the code run faster</label><br>
<input type="radio" name="eq4" id="eq4c"><label for="eq4c"> C) It's purely cosmetic and doesn't change the values</label><br>

<br>
<details><summary style="cursor:pointer; background:#4A90D9; color:white; padding:8px 16px; border-radius:6px; display:inline-block;"><b>📝 Check Answers</b></summary>
<div style="margin-top:12px; padding:12px; background:#e8f5e9; border-radius:8px;">
✅ <b>Q1: B)</b> room_type has a real business order (Standard < Deluxe < Suite), so a single label column (0, 1, 2) lets models use that ordering directly.<br><br>
✅ <b>Q2: A)</b> meal_plan has no natural order and only 4 values, so one-hot is cheap and avoids implying a false ranking between categories.<br><br>
✅ <b>Q3: B)</b> One-hot on 54 countries would add 54 sparse columns (most guests come from a handful of countries); target encoding compresses this into one informative numeric column.<br><br>
✅ <b>Q4: A)</b> Without smoothing, a country with just 2 bookings (both canceled) would get target-encoded as 1.0, an extreme, unreliable estimate. Smoothing blends the category mean with the global mean, weighted by how many samples support it.
</div></details>
</div>
'''
display(HTML(quiz_html))

## 2. 🧹 Data Preprocessing

Now that we've seen the data, let's clean it. **For each step, answer the question first, then run the code.**

In [ ]:
#@title Q8: Preprocessing decisions (answer ALL before running the code below)
quiz_html = '''
<div style="border:2px solid #4A90D9; border-radius:12px; padding:20px; margin:10px 0; background:#f0f7ff;">
<h4>🧠 Preprocessing Decisions: answer these based on what you saw in the EDA:</h4>

<p><b>Q1: DATA LEAK: Which column must be dropped?</b></p>
<input type="radio" name="pp1" id="pp1a"><label for="pp1a"> A) booking_id</label><br>
<input type="radio" name="pp1" id="pp1b"><label for="pp1b"> B) cancellation_fee_charged</label><br>
<input type="radio" name="pp1" id="pp1c"><label for="pp1c"> C) Both A and B</label><br>
<input type="radio" name="pp1" id="pp1d"><label for="pp1d"> D) Neither</label><br>

<p><b>Q2: MISSING VALUES: How should we impute <code>children</code> (5% missing)?</b></p>
<input type="radio" name="pp2" id="pp2a"><label for="pp2a"> A) Mean (≈0.4 children)</label><br>
<input type="radio" name="pp2" id="pp2b"><label for="pp2b"> B) Median (0 children)</label><br>
<input type="radio" name="pp2" id="pp2c"><label for="pp2c"> C) Mode = 0 (most common)</label><br>
<input type="radio" name="pp2" id="pp2d"><label for="pp2d"> D) Drop all rows with missing children</label><br>

<p><b>Q3: MISSING VALUES: How should we handle <code>previous_cancellations</code> (17% missing)?</b></p>
<input type="radio" name="pp3" id="pp3a"><label for="pp3a"> A) Impute with mean</label><br>
<input type="radio" name="pp3" id="pp3b"><label for="pp3b"> B) Impute with 0 (logical value for first-time guests)</label><br>
<input type="radio" name="pp3" id="pp3c"><label for="pp3c"> C) Impute with 0 AND create a "was_missing" flag</label><br>
<input type="radio" name="pp3" id="pp3d"><label for="pp3d"> D) Drop the column entirely</label><br>

<p><b>Q4: DUPLICATES: What should we do with duplicate rows?</b></p>
<input type="radio" name="pp4" id="pp4a"><label for="pp4a"> A) Keep them (they might be real bookings)</label><br>
<input type="radio" name="pp4" id="pp4b"><label for="pp4b"> B) Drop duplicates (they add no new information)</label><br>

<p><b>Q5: OUTLIERS: avg_daily_rate has values > €1000. How should we handle them?</b></p>
<input type="radio" name="pp5" id="pp5a"><label for="pp5a"> A) Remove those rows</label><br>
<input type="radio" name="pp5" id="pp5b"><label for="pp5b"> B) Clip at the 99th percentile</label><br>
<input type="radio" name="pp5" id="pp5c"><label for="pp5c"> C) Do nothing</label><br>

<br>
<details><summary style="cursor:pointer; background:#4A90D9; color:white; padding:8px 16px; border-radius:6px; display:inline-block;"><b>📝 Check All Answers</b></summary>
<div style="margin-top:12px; padding:12px; background:#e8f5e9; border-radius:8px;">
✅ <b>Q1: C)</b> Drop BOTH: booking_id is a unique row label with no predictive value (causes memorization), and cancellation_fee_charged is a data leak.<br><br>
✅ <b>Q2: C)</b> Mode = 0. You can't have 0.4 children. The mode (most common value) is 0, which is the most sensible imputation for a count variable.<br><br>
✅ <b>Q3: C)</b> Impute with 0 AND add a flag. The data is MNAR: missing for first-time guests who logically have 0 cancellations. The flag preserves the "new guest" signal. Using the mean (~0.7) would wrongly imply first-timers have cancellation history.<br><br>
✅ <b>Q4: B)</b> Drop them. Exact duplicates (same values in every column except booking_id) add no new information and can bias the model.<br><br>
✅ <b>Q5: B)</b> Clip at 99th percentile. Removing rows loses all other columns. Clipping preserves the "expensive booking" signal without letting extreme values distort training. A €1,200 booking isn't an error; it's a luxury suite.
</div></details>
</div>
'''
display(HTML(quiz_html))

In [ ]:
#@title Now the code implements exactly what you selected above:
df = train.copy(); test_df = test.copy()
TARGET = 'is_canceled'; y = df[TARGET].copy()

# Q1: Drop leak + ID
df = df.drop(columns=['cancellation_fee_charged', 'booking_id'])
test_df = test_df.drop(columns=['cancellation_fee_charged', 'booking_id'])
print("✓ Dropped cancellation_fee_charged (leak) and booking_id (no predictive value)")

# Q2: children → fill with 0
df['children'] = df['children'].fillna(0)
test_df['children'] = test_df['children'].fillna(0)
print("✓ children: imputed with 0 (mode)")

# Q3: previous_cancellations → fill with 0 + flag
df['prev_cancel_missing'] = df['previous_cancellations'].isnull().astype(int)
test_df['prev_cancel_missing'] = test_df['previous_cancellations'].isnull().astype(int)
df['previous_cancellations'] = df['previous_cancellations'].fillna(0)
test_df['previous_cancellations'] = test_df['previous_cancellations'].fillna(0)
print("✓ previous_cancellations: imputed with 0 + created prev_cancel_missing flag")

# Q4: Drop duplicates
before = len(df)
df = df.drop_duplicates(subset=df.columns.difference([TARGET, 'arrival_date', 'booking_date']).tolist())
y = df[TARGET]
print(f"✓ Removed {before - len(df)} duplicate rows")

# Q5: Clip outliers
q99 = df['avg_daily_rate'].quantile(0.99)
df['avg_daily_rate'] = df['avg_daily_rate'].clip(upper=q99)
test_df['avg_daily_rate'] = test_df['avg_daily_rate'].clip(upper=q99)
print(f"✓ Clipped avg_daily_rate at 99th percentile (€{q99:.0f})")

In [ ]:
#@title Encode categoricals
room_map = {'Standard': 0, 'Deluxe': 1, 'Suite': 2}
df['room_type_enc'] = df['room_type'].map(room_map)
test_df['room_type_enc'] = test_df['room_type'].map(room_map)

low_card = ['meal_plan', 'market_segment', 'deposit_type']
df = pd.get_dummies(df, columns=low_card, drop_first=True)
test_df = pd.get_dummies(test_df, columns=low_card, drop_first=True)
for c in df.columns:
    if c not in test_df.columns and c != TARGET: test_df[c] = 0
for c in test_df.columns:
    if c not in df.columns: df[c] = 0

# Target encoding for country (with smoothing)
country_means = df.groupby('country')[TARGET].mean()
global_mean = y.mean(); country_counts = df['country'].value_counts(); smoothing = 20
country_target_enc = (country_counts * country_means + smoothing * global_mean) / (country_counts + smoothing)
df['country_enc'] = df['country'].map(country_target_enc)
test_df['country_enc'] = test_df['country'].map(country_target_enc).fillna(global_mean)
df = df.drop(columns=['country', 'room_type']); test_df = test_df.drop(columns=['country', 'room_type'])

print(f"✓ Encoded: room_type (label), meal_plan/market_segment/deposit_type (one-hot), country (target)")

# Measure
feature_cols_pp = [c for c in df.columns if c not in [TARGET,'booking_date','arrival_date'] and df[c].dtype in ['int64','float64','uint8','bool']]
pp_scores = cross_val_score(ruler, df[feature_cols_pp], y, cv=cv, scoring='roc_auc')
log_score('2', 'After preprocessing', pp_scores.mean())

## 3. 🛠️ Feature Engineering

In real-world projects, what can make the difference between a successful machine learning model and a mediocre one is often the data, not the model. The real differentiator is the informational value of the content itself, which is represented by the type and quality of the features. Feature engineering, the set of techniques for transforming data into more useful information for your models, is invariably a key driver of performance in competitions. Even the most powerful model still needs the data processed and rendered into a more understandable form.

Below we apply a few standard families of feature engineering techniques:

- **Time feature processing:** splitting a date into its elements (year, month, day), transforming it into week of the year or weekday, computing differences between dates, and, when available, computing differences with key events such as holidays. In this notebook we extract `booking_month`, `arrival_month`, and a weekend-arrival flag; the same idea extends naturally to week-of-year or days-to-holiday features on a real project.
- **Categorical feature encoding:** covered in the previous section (label, one-hot, target encoding).
- **Aggregation features:** combine related columns into higher-level concepts, such as `total_guests` (adults + children + babies) and `total_stays` (weekend + weekday nights).
- **Interaction features:** capture relationships between features that the model can't easily discover on its own, such as `rate_per_guest` (price per person) or `requests_per_stay`.

We also apply the **log transforms** identified in the EDA section.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🛠️ Feature Engineering — YOUR TURN!
# Create all features for BOTH df and test_df.
# Loop over both frames so train and test get the same columns.
# ═══════════════════════════════════════════════════════════════

for frame in [df, test_df]:

    # ── Datetime features ─────────────────────────────────────
    # 🎯 Extract the booking month (1–12) from 'booking_date'
    frame['booking_month'] = 🎯

    # 🎯 Extract the arrival month (1–12) from 'arrival_date'
    frame['arrival_month'] = 🎯

    # 🎯 Binary flag: 1 if arrival is Saturday or Sunday, else 0
    # Hint: .dt.dayofweek gives 0=Mon … 6=Sun; weekend is >= 5
    frame['is_weekend_arrival'] = 🎯

    # ── Aggregation features ──────────────────────────────────
    # 🎯 Total guests (adults + children + babies)
    frame['total_guests'] = 🎯

    # 🎯 Total nights (weekend + weekday)
    frame['total_stays'] = 🎯

    # 🎯 Estimated total cost (total_stays × avg_daily_rate)
    frame['total_cost'] = 🎯

    # ── Interaction features ──────────────────────────────────
    # 🎯 Price per guest: avg_daily_rate / total_guests
    # Hint: .clip(1) on the denominator avoids division by zero
    frame['rate_per_guest'] = 🎯

    # 🎯 Special requests per night: special_requests / total_stays
    frame['requests_per_stay'] = 🎯

    # 🎯 Is this a family? (children > 0 OR babies > 0), cast to int
    frame['is_family'] = 🎯

    # ── Log transforms (lead_time skew=1.80, avg_daily_rate skew=6.40) ──
    # 🎯 np.log1p(lead_time)
    frame['lead_time_log'] = 🎯

    # 🎯 np.log1p(avg_daily_rate)
    frame['avg_daily_rate_log'] = 🎯

# ── Agent target encoding (provided) ─────────────────────────
am = df.groupby('agent_id')[TARGET].mean(); ac = df['agent_id'].value_counts()
ae = (ac * am + smoothing * global_mean) / (ac + smoothing)
df['agent_cancel_rate'] = df['agent_id'].map(ae).fillna(global_mean)
test_df['agent_cancel_rate'] = test_df['agent_id'].map(ae).fillna(global_mean)

df = df.drop(columns=['booking_date', 'arrival_date']); test_df = test_df.drop(columns=['booking_date', 'arrival_date'])
print(f"Created features incl. log transforms: lead_time_log, avg_daily_rate_log")
print(f"Total features: {df.shape[1] - 1}")

fe_cols = [c for c in df.columns if c != TARGET and df[c].dtype in ['int64','float64','uint8','bool']]
fe_scores = cross_val_score(ruler, df[fe_cols], y, cv=cv, scoring='roc_auc')
log_score('3', 'After feature engineering', fe_scores.mean())

## 4. 🎯 Feature Selection

Not all features help, some add noise. If our score didn't improve after feature engineering, that's a signal that we added redundant or noisy features. Let's keep only the useful ones.

**Methods:** correlation with target, Random Forest importance, and optionally [Recursive Feature Elimination (RFE)](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.RFE.html).

See also: [scikit-learn Feature Selection guide](https://scikit-learn.org/stable/modules/feature_selection.html)

**Why correlation with target?** It's a fast, cheap first pass: for each numeric feature, measure the (linear) correlation with the target. It only captures linear relationships and ignores interactions, but it's a quick way to spot features with essentially zero signal.

This is the **Pearson correlation coefficient**: it ranges from $-1$ (perfect negative linear relationship) to $+1$ (perfect positive linear relationship), with $0$ meaning no *linear* relationship. `X.corrwith(y)` computes exactly this, feature by feature.

$$\rho_{X,y} = \frac{\operatorname{Cov}(X, y)}{\sigma_X\, \sigma_y} = \frac{\sum_i (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_i (x_i - \bar{x})^2}\;\sqrt{\sum_i (y_i - \bar{y})^2}}$$

**Why Random Forest importance?** Unlike correlation, it captures non-linear relationships and interactions, because it measures how much each feature actually reduces impurity across many trees trained on random subsets of the data and features. It's a better proxy for "does this feature help the model" than a raw correlation number.



In [ ]:
# 4.1 Correlation with target
feature_cols = [c for c in df.columns if c != TARGET and df[c].dtype in ['int64','float64','uint8','bool']]
X = df[feature_cols]

# 🎯 Compute absolute Pearson correlation of each feature with y,
#    sorted descending (strongest first).
# Hint: X.corrwith(y).abs().sort_values(ascending=False)
corr_with_target = 🎯

print("Top 15 features (absolute correlation with target):")
print(corr_with_target.head(15).round(4))

In [ ]:
# 4.2 Random Forest importance
from sklearn.ensemble import RandomForestClassifier
feature_cols = [c for c in df.columns if c != TARGET and df[c].dtype in ['int64','float64','uint8','bool']]
X = df[feature_cols]

# 🎯 Create a Random Forest: 100 trees, max_depth=8, random_state=42, n_jobs=-1
rf_quick = 🎯

# 🎯 Fit on X, y
rf_quick.🎯

# 🎯 Feature importances as a pd.Series(rf_quick.feature_importances_, index=feature_cols)
#    sorted ascending (.sort_values(ascending=True)) for the bar plot
importances = 🎯  # avg impurity decrease per feature across all trees

fig, ax = plt.subplots(figsize=(8, max(6, len(feature_cols)*0.22)))
importances.tail(20).plot.barh(ax=ax, color='#4A90D9')
ax.set_title('Top 20 Feature Importances'); plt.tight_layout(); plt.show()

selected = importances[importances > 0.005].index.tolist()
dropped = importances[importances <= 0.005].index.tolist()
print(f"Selected: {len(selected)}, Dropped: {len(dropped)} → {dropped}")
X = df[selected]
X_test = test_df[[c for c in selected if c in test_df.columns]]
for c in selected:
    if c not in X_test.columns: X_test[c] = 0

## 5. ✅ Validation Strategy

How you split your data determines how trustworthy your scores are.

- **Train/Test Split**: reserve 20% for test. Fast, but the score depends on the random split.
- **K-Fold Cross-Validation**: split into K parts, train on K-1, validate on 1, rotate K times. Average K scores, much more robust.
- **Stratified K-Fold**: like K-Fold, but each fold preserves the class ratio (~37% canceled). Prevents misleading scores from random class imbalance in a fold. Use when the target is imbalanced (our case!).

All scores in the performance tracker use **5-fold Stratified K-Fold** on the training set. We never touch the test set during model selection.

In [ ]:
#@title ⚖️ 5.1 Compare Validation Strategies
from sklearn.model_selection import KFold, StratifiedKFold
quick = GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42)

kf  = KFold(n_splits=5, shuffle=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

s_kf  = cross_val_score(quick, X, y, cv=kf,  scoring='roc_auc')
s_skf = cross_val_score(quick, X, y, cv=skf, scoring='roc_auc')

print("VALIDATION COMPARISON")
print(f"  K-Fold:            {s_kf.mean():.4f} ± {s_kf.std():.4f}")
print(f"  Stratified K-Fold: {s_skf.mean():.4f} ± {s_skf.std():.4f}")
print("\nStratified K-Fold usually gives lower variance (more stable, more reliable).")

In [ ]:
#@title Q9: Choosing a Validation Strategy
quiz_html = '''
<div style="border:2px solid #4A90D9; border-radius:12px; padding:20px; margin:10px 0; background:#f0f7ff;">
<h4>🧠 Choosing a validation strategy</h4>

<p><b>Q1: Why not just use a single train/test split to compare models here?</b></p>
<input type="radio" name="vq1" id="vq1a"><label for="vq1a"> A) It's the best method, always use it</label><br>
<input type="radio" name="vq1" id="vq1b"><label for="vq1b"> B) A single split gives one noisy estimate; a different random split could give a meaningfully different score</label><br>
<input type="radio" name="vq1" id="vq1c"><label for="vq1c"> C) It's slower than K-Fold</label><br>

<p><b>Q2: With a target that's ~37% positive, why prefer Stratified K-Fold over plain K-Fold?</b></p>
<input type="radio" name="vq2" id="vq2a"><label for="vq2a"> A) Plain K-Fold could by chance create folds with very different class balance, making scores noisier and less comparable</label><br>
<input type="radio" name="vq2" id="vq2b"><label for="vq2b"> B) Stratified K-Fold is always faster to run</label><br>
<input type="radio" name="vq2" id="vq2c"><label for="vq2c"> C) There is no real difference between the two</label><br>

<p><b>Q3: What does averaging 5 fold scores give us that a single fold score doesn't?</b></p>
<input type="radio" name="vq3" id="vq3a"><label for="vq3a"> A) A less noisy, more robust estimate of performance on unseen data (plus a standard deviation, telling us how stable that estimate is)</label><br>
<input type="radio" name="vq3" id="vq3b"><label for="vq3b"> B) A guaranteed higher score</label><br>
<input type="radio" name="vq3" id="vq3c"><label for="vq3c"> C) Nothing, it's the exact same information as one fold</label><br>

<br>
<details><summary style="cursor:pointer; background:#4A90D9; color:white; padding:8px 16px; border-radius:6px; display:inline-block;"><b>📝 Check Answers</b></summary>
<div style="margin-top:12px; padding:12px; background:#e8f5e9; border-radius:8px;">
✅ <b>Q1: B)</b> One split is one draw of the dice. Two equally good models can look meaningfully different just because of which rows happened to land in the test set.<br><br>
✅ <b>Q2: A)</b> With an imbalanced target, plain K-Fold can accidentally create a fold with, say, 25% or 50% positives instead of ~37%, which distorts the score for that fold. Stratified K-Fold forces every fold to keep the same ~37% ratio.<br><br>
✅ <b>Q3: A)</b> Averaging across folds smooths out the luck of any single split, and the spread (standard deviation) across folds tells you how much to trust that average.
</div></details>
</div>
'''
display(HTML(quiz_html))

## 6. 📐 Baseline Model

Always start simple. The baseline tells you: is the problem learnable? How much room for improvement?
 A baseline is also the yardstick for everything that follows: without it, you can't tell whether a fancier model, extra features, or hours of tuning are actually earning their complexity, or just adding noise.


In [ ]:
#@title 6.1 Logistic Regression (needs scaling)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lr_pipe = Pipeline([('scaler', StandardScaler()),
                    ('lr', LogisticRegression(max_iter=1000, random_state=42))])
lr_scores = cross_val_score(lr_pipe, X, y, cv=cv, scoring='roc_auc')
print(f"Logistic Regression: AUC = {lr_scores.mean():.4f} ± {lr_scores.std():.4f}")
log_score('6a', 'Logistic Regression (baseline)', lr_scores.mean())

In [ ]:
#@title 6.2 Decision Tree
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_scores = cross_val_score(dt, X, y, cv=cv, scoring='roc_auc')
print(f"Decision Tree (d=5): AUC = {dt_scores.mean():.4f} ± {dt_scores.std():.4f}")
log_score('6b', 'Decision Tree (depth=5)', dt_scores.mean())

## 7. ⚔️ Compare Models: From Simple to Powerful

Our baseline models (LogReg, Decision Tree) are single models. Now we move to ensemble methods, models that combine many weaker learners:

- **Random Forest** = many independent trees, averaged (bagging)
- **XGBoost / LightGBM / CatBoost** = many trees trained sequentially, each correcting the previous one's mistakes (gradient boosting)

All of these are ensemble methods internally. In Section 9, we'll go one level further: combining DIFFERENT ensemble models together (meta-ensembling).

### Gradient Boosting: Key Idea

1. Start with a simple prediction (e.g. the average cancel rate)
2. Train a small tree to predict the errors (residuals)
3. Add that tree's predictions (scaled by `learning_rate`) to the ensemble
4. Repeat, each tree fixes the previous ensemble's mistakes

Key hyperparameters: `n_estimators` (number of trees), `max_depth` (tree complexity), `learning_rate` (step size, smaller = more trees needed but better generalization).

In [ ]:
#@title 7.1 Compare Models: Simple → Powerful
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

models = {
    'Random Forest': RandomForestClassifier(n_estimators=300, max_depth=10, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, random_state=42, eval_metric='logloss', verbosity=0),
    'LightGBM': LGBMClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, random_state=42, verbose=-1),
    'CatBoost': CatBoostClassifier(iterations=300, depth=6, learning_rate=0.1, random_state=42, verbose=0),
}
results = {}
for name, model in models.items():
    s = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')
    results[name] = {'mean': s.mean(), 'std': s.std()}
    print(f"{name:20s}: {s.mean():.4f} ± {s.std():.4f}")
best = max(results, key=lambda k: results[k]['mean'])
log_score('7', f'Best: {best}', results[best]['mean'])

In [ ]:
#@title 7.2 Visual Comparison
fig, ax = plt.subplots(figsize=(10, 4))
names = list(results.keys()); means = [results[n]['mean'] for n in names]; stds = [results[n]['std'] for n in names]
bars = ax.barh(names, means, xerr=stds, color='#4A90D9', edgecolor='white', capsize=5)
ax.set_xlabel('ROC-AUC'); ax.set_title('Model Comparison (5-fold Stratified CV)')
ax.set_xlim(min(means)-0.05, max(means)+0.02)
for bar, m, s in zip(bars, means, stds):
    ax.text(m+s+0.003, bar.get_y()+bar.get_height()/2, f'{m:.4f}', va='center')
plt.tight_layout(); plt.show()

## 8. ⚙️ Hyperparameter Optimization (pre-computed)

We ran Grid Search and Optuna for you. Here are the results, use them directly.

### Grid Search

Grid search exhaustively tries every combination of hyperparameter values you specify: for every parameter, you pick a set of values to test, and the grid is the full cross-product of all of them. For example, testing 3 values of `n_estimators`, 3 of `max_depth`, and 2 of `learning_rate` means 3 × 3 × 2 = 18 full model fits (times the number of CV folds). Add a fourth parameter with 4 values and you're at 72 fits. This explosive growth in the number of combinations as you add more parameters is called the **curse of dimensionality**, and it's what makes grid search infeasible in high-dimensional hyperparameter spaces.

On the positive side, grid search is [embarrassingly parallel](https://www.cs.iusb.edu/~danav/teach/b424/b424_23_embpar.html): each combination can be evaluated completely independently of the others. This means you can get an answer very quickly if you have enough processors to spread the search across.

A subtler downside: grid search **forgets information**. Each combination is evaluated in isolation, so the result of combination #5 tells the search nothing about where to look next; time is wasted exhaustively covering regions that are obviously bad.

### Grid Search result (LightGBM, 12 combinations):
```
Best params: {'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 200}
Best AUC:    0.7882
```

### Optuna

Optuna is a Bayesian-style hyperparameter optimizer. Instead of blindly evaluating every combination, it builds a probabilistic model of "which regions of the hyperparameter space tend to score well" from the trials it has already run, and uses that model to pick the next combination to try. Its search is *informed*, not uninformative: it spends more trials in promising regions and fewer in regions that already looked bad, so it tends to reach strong hyperparameters in far fewer trials than an exhaustive grid.

### Optuna results (50 trials per model):

In [ ]:
# 8.1 Pre-computed Best Hyperparameters
# Optuna found these (50 trials per model). Instantiate each, then CV.

best_lgbm_params = {'n_estimators': 500, 'max_depth': 7, 'learning_rate': 0.05,
    'num_leaves': 63, 'min_child_samples': 20, 'subsample': 0.8, 'colsample_bytree': 0.8,
    'reg_alpha': 0.1, 'reg_lambda': 1.0}

best_xgb_params = {'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.05,
    'min_child_weight': 5, 'subsample': 0.8, 'colsample_bytree': 0.8,
    'gamma': 0.1, 'reg_alpha': 0.1, 'reg_lambda': 1.0}

best_cb_params = {'iterations': 500, 'depth': 7, 'learning_rate': 0.05,
    'l2_leaf_reg': 3.0, 'bagging_temperature': 0.5}

# 🎯 Instantiate tuned LightGBM
# Hint: LGBMClassifier(**best_lgbm_params, random_state=42, verbose=-1)
tuned_lgbm = #🎯

# 🎯 Instantiate tuned XGBoost
# Hint: XGBClassifier(**best_xgb_params, random_state=42, eval_metric='logloss', verbosity=0)
tuned_xgb  = #🎯

# 🎯 Instantiate tuned CatBoost
# Hint: CatBoostClassifier(**best_cb_params, random_state=42, verbose=0)
tuned_cb   = #🎯

# 🎯 Cross-validate each model. Fill in the sklearn function name.
# Hint: cross_val_score(model, X, y, cv=cv, scoring='roc_auc')
for name, m in [('Tuned LGBM', tuned_lgbm), ('Tuned XGB', tuned_xgb), ('Tuned CB', tuned_cb)]:
    s = #🎯(m, X, y, cv=cv, scoring='roc_auc')
    print(f"{name}: AUC = {s.mean():.4f} ± {s.std():.4f}")

log_score('8', 'Tuned LGBM', cross_val_score(tuned_lgbm, X, y, cv=cv, scoring='roc_auc').mean())

<details><summary><b>📝 Grid Search reference code</b></summary>

```python
from sklearn.model_selection import GridSearchCV
param_grid = {'n_estimators': [200, 400], 'max_depth': [4, 6, 8], 'learning_rate': [0.05, 0.1]}
gs = GridSearchCV(LGBMClassifier(random_state=42, verbose=-1), param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
gs.fit(X, y)
print(f"Best: {gs.best_params_}, AUC: {gs.best_score_:.4f}")
```

</details>

<details><summary><b>📝 Optuna reference code</b></summary>

```python
import optuna
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 800),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 127),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
    }
    return cross_val_score(LGBMClassifier(**params, random_state=42, verbose=-1),
                           X, y, cv=cv, scoring='roc_auc').mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)
```

</details>

## 9. 🧩 Ensemble Methods

You already know what ensembles are (bagging, boosting, stacking) from the previous exercise, so here's the one-line recap: **combine multiple models to reduce variance and improve robustness.** The key idea: diverse models make different errors, and combining them averages out individual mistakes.

Now let's apply meta-ensembling: combining **different tuned models** together.

In [ ]:
#@title 9.1 Build Tuned Base Models (Out-of-Fold Predictions)
from sklearn.base import clone
from sklearn.metrics import roc_auc_score

tuned_rf = RandomForestClassifier(n_estimators=500, max_depth=12, random_state=42, n_jobs=-1)
tuned_lr = Pipeline([('s', StandardScaler()), ('l', LogisticRegression(C=0.5, max_iter=1000))])
base_models = {'LGBM': tuned_lgbm, 'XGB': tuned_xgb, 'CB': tuned_cb, 'RF': tuned_rf, 'LR': tuned_lr}

oof = pd.DataFrame(index=X.index)
test_p = pd.DataFrame(index=X_test.index)
print("Building tuned base models from Optuna results...")
for name, model in base_models.items():
    o = np.zeros(len(X)); tp = np.zeros((len(X_test), 5))
    for fi, (tri, vi) in enumerate(cv.split(X, y)):
        m = clone(model); m.fit(X.iloc[tri], y.iloc[tri])
        o[vi] = m.predict_proba(X.iloc[vi])[:, 1] if hasattr(m, 'predict_proba') else m.decision_function(X.iloc[vi])
        tp[:, fi] = m.predict_proba(X_test)[:, 1] if hasattr(m, 'predict_proba') else m.decision_function(X_test)
    oof[name] = o; test_p[name] = tp.mean(axis=1)
    print(f"  {name:9s}: AUC = {roc_auc_score(y, oof[name]):.4f}")

In [ ]:
#@title 9.2 Simple Averaging
avg_all = oof.mean(axis=1)
avg_gbm = oof[['LGBM','XGB','CB']].mean(axis=1)
avg_gbm_rf = oof[['LGBM','XGB','CB','RF']].mean(axis=1)
print(f"Simple Average (all 5 models): AUC = {roc_auc_score(y, avg_all):.4f}")
print(f"Simple Average (3 GBMs only):  AUC = {roc_auc_score(y, avg_gbm):.4f}")
print(f"Simple Average (3 GBMs + RF):  AUC = {roc_auc_score(y, avg_gbm_rf):.4f}")
log_score('9a', 'Simple Average (all 5)', roc_auc_score(y, avg_all))

Simple averaging treats every model equally. But some models are clearly stronger than others here (CatBoost > XGBoost > LightGBM > RF ≈ LogReg), so it's natural to ask: can we do better by weighting the stronger models more?

We optimize the weights directly against the metric we care about (ROC-AUC) using Optuna. This runs in about a second, because we're only tuning 5 numbers against already-computed OOF predictions, not retraining any models.

In [ ]:
#@title 9.3 Optimized Weighted Averaging
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def weight_objective(trial):
    w = np.array([trial.suggest_float(f'w_{c.lower()}', 0, 1) for c in oof.columns])
    if w.sum() == 0:
        return 0.0
    w = w / w.sum()
    blend = (oof.values * w).sum(axis=1)
    return roc_auc_score(y, blend)

study_w = optuna.create_study(direction='maximize')
study_w.optimize(weight_objective, n_trials=100, show_progress_bar=True)

w = np.array([study_w.best_params[f'w_{c.lower()}'] for c in oof.columns])
w = w / w.sum()
print("Best weights (normalized):")
for c, wi in zip(oof.columns, w):
    print(f"  w_{c.lower()}: {wi:.3f}")

weighted_auc = roc_auc_score(y, (oof.values * w).sum(axis=1))
print(f"\nWeighted Average AUC: {weighted_auc:.4f}")
log_score('9b', 'Weighted Average (Optuna-optimized)', weighted_auc)

### 📌 A note on stacking

We also tried **stacking** (training a meta-model, Logistic Regression, on the base models' predictions via `StackingClassifier`). On this dataset it lands close to the weighted average (AUC ≈ 0.799), but costs much more compute, since `StackingClassifier` retrains all 5 base models internally for every one of its own CV folds. Because this is a 1-hour exercise, we run the fast, high-scoring weighted average live above, and keep the stacking code below as a reference in case you want to try it later.

<details><summary><b>📝 Stacking reference code (not run live, takes several minutes)</b></summary>

```python
from sklearn.ensemble import StackingClassifier

stacking = StackingClassifier(
    estimators=[('lgbm', tuned_lgbm), ('xgb', tuned_xgb), ('cb', tuned_cb),
                ('rf', tuned_rf), ('lr', tuned_lr)],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5,
    n_jobs=-1
)
st_s = cross_val_score(stacking, X, y, cv=cv, scoring='roc_auc')
print(f"Stacking: AUC = {st_s.mean():.4f} ± {st_s.std():.4f}")
```

Measured results from a full run:
```
Stacking (5 diverse models -> LR meta):            AUC = 0.7992 ± 0.0072
Stacking (5 diverse -> LGBM meta + passthrough):    AUC = 0.7985 ± 0.0067
```

</details>

### ⭐ Bonus: a more different base model (SVM)

All five base models so far are either trees (LGBM, XGB, CB, RF) or a linear model (LogReg). The tuned GBMs are all correlated with each other, since they're all boosted/bagged trees, so they tend to make *similar* mistakes. A Support Vector Machine with an RBF kernel draws smooth, curved decision boundaries instead of axis-aligned splits, so it tends to get *different* rows wrong. That diversity is exactly what an ensemble can exploit: adding a model that's wrong in a different way is more valuable than adding one more model that's wrong in the same way.

In [ ]:
#@title ⭐ Bonus: SVM as a Base Model
from sklearn.svm import SVC

tuned_svm = Pipeline([('s', StandardScaler()), ('svm', SVC(probability=True, random_state=42))])
o_svm = np.zeros(len(X)); tp_svm = np.zeros((len(X_test), 5))
for fi, (tri, vi) in enumerate(cv.split(X, y)):
    m = clone(tuned_svm); m.fit(X.iloc[tri], y.iloc[tri])
    o_svm[vi] = m.predict_proba(X.iloc[vi])[:, 1]
    tp_svm[:, fi] = m.predict_proba(X_test)[:, 1]
oof['SVM'] = o_svm
test_p['SVM'] = tp_svm.mean(axis=1)

print(f"SVM (RBF kernel): AUC = {roc_auc_score(y, oof['SVM']):.4f}")
print(f"\nCorrelation between each model's OOF predictions (lower = more diverse):")
print(oof.corr().round(2))
log_score('9c', 'SVM (bonus base model)', roc_auc_score(y, oof['SVM']))


### ⭐ Bonus: Dynamic weights (Feature-Weighted Linear Stacking)

Our weighted average in 9.3 used **one fixed set of weights for every booking**. We saw in EDA that `deposit_type` splits guests into very different behavior groups (No Deposit bookings cancel at ~45%, Non-Refundable bookings at ~15%). It's plausible that which model is *most reliable* also differs between these groups, not just the cancellation rate itself.

**Feature-Weighted Linear Stacking (FWLS)** lets each base model's weight depend on a feature, instead of being one fixed number. The trick: alongside each base model's prediction, we also feed the meta-model that same prediction *multiplied by* a gate feature (here, "is this a Non-Refundable booking?"). With a linear meta-model, the effective weight on each base model becomes:

$$\text{weight}_k(x) = a_k + b_k \cdot \text{gate}(x)$$

so the blend can lean on a different model for No-Deposit bookings than for Non-Refundable ones, instead of compromising with one weighting everywhere.

In [ ]:
#@title ⭐ Bonus: Dynamic Weights by Deposit Type
from sklearn.linear_model import LinearRegression

gate_col = 'deposit_type_Non Refund'
assert gate_col in df.columns, "deposit_type wasn't one-hot encoded as expected"

gate = df.loc[X.index, gate_col].to_numpy().reshape(-1, 1).astype(float)
oof_matrix = oof.values

def augment(P, G):
    return np.column_stack([P] + [P[:, [k]] * G for k in range(P.shape[1])])

static_meta = LinearRegression().fit(oof_matrix, y)
static_auc = roc_auc_score(y, static_meta.predict(oof_matrix))

dyn_meta = LinearRegression().fit(augment(oof_matrix, gate), y)
dynamic_auc = roc_auc_score(y, dyn_meta.predict(augment(oof_matrix, gate)))

print(f"Static stack   AUC = {static_auc:.4f}  (one weighting for every booking)")
print(f"Dynamic stack  AUC = {dynamic_auc:.4f}  (weights depend on deposit_type)")
print(f"Lift = {dynamic_auc - static_auc:+.4f} AUC")

n_models = oof.shape[1]
base_w = dyn_meta.coef_[:n_models]
inter_w = dyn_meta.coef_[n_models:]
print("\nEffective weight on each model, by deposit type:")
print(f"  {'Model':9s}  {'No Deposit / Refundable':>24s}  {'Non-Refundable':>15s}")
for i, name in enumerate(oof.columns):
    w_base = base_w[i]
    w_non_refund = base_w[i] + inter_w[i]
    print(f"  {name:9s}  {w_base:+24.3f}  {w_non_refund:+15.3f}")

In [ ]:
#@title ⭐ CV estimate for the dynamic stack (cross-validates the meta-step itself)
meta_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
dyn_oof = np.zeros(len(y))
for tri, vi in meta_cv.split(oof_matrix, y):
    m = LinearRegression().fit(augment(oof_matrix[tri], gate[tri]), y.iloc[tri])
    dyn_oof[vi] = m.predict(augment(oof_matrix[vi], gate[vi]))

dynamic_auc_honest = roc_auc_score(y, dyn_oof)
print(f"Dynamic stack (honest, meta-model properly cross-validated): AUC = {dynamic_auc_honest:.4f}")
log_score('9d', 'Dynamic Stack (FWLS, deposit_type gate)', dynamic_auc_honest)

> **Which model is this?** The error analysis below is run on the single **tuned LightGBM** model (`best_lgbm_params` from Section 8), not the ensembles from Section 9. A lone model is far easier to diagnose, one confusion matrix and one ROC curve, than a weighted blend of six models, so it's the more useful lens for asking *"where exactly is the model getting it wrong, and why?"*


In [ ]:
#@title 10. Error Analysis & Kaggle Submission
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
final = LGBMClassifier(**best_lgbm_params, random_state=42, verbose=-1)
final.fit(X_tr, y_tr)
y_proba = final.predict_proba(X_val)[:, 1]
print(classification_report(y_val, (y_proba>=0.5).astype(int), target_names=['Not Canceled', 'Canceled']))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
cm = confusion_matrix(y_val, (y_proba>=0.5).astype(int))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Not','Cancel'], yticklabels=['Not','Cancel'])
axes[0].set_ylabel('Actual'); axes[0].set_xlabel('Predicted')
fpr, tpr, _ = roc_curve(y_val, y_proba)
axes[1].plot(fpr, tpr, 'b-', lw=2, label=f'AUC={roc_auc_score(y_val, y_proba):.4f}')
axes[1].plot([0,1],[0,1],'k--'); axes[1].legend(); axes[1].set_title('ROC Curve')
plt.tight_layout(); plt.show()

In [ ]:
#@title Q10: Interpreting the Error Analysis
quiz_html = '''
<div style="border:2px solid #E8593C; border-radius:12px; padding:20px; margin:10px 0; background:#fff5f5;">
<h4>📊 Based on the classification report and plots above:</h4>

<p><b>Q1: The model achieves 81%% recall on "Not Canceled" but only 53%% recall on "Canceled." What does this mean in business terms?</b></p>
<input type="radio" name="ea1" id="ea1a"><label for="ea1a"> A) The model catches most guests who will show up, but misses nearly half of the guests who will cancel</label><br>
<input type="radio" name="ea1" id="ea1b"><label for="ea1b"> B) The model is better at predicting cancellations than non-cancellations</label><br>
<input type="radio" name="ea1" id="ea1c"><label for="ea1c"> C) 53%% recall means the model is worse than random guessing for cancellations</label><br>

<p><b>Q2: Look at the confusion matrix. The model predicts "Not Canceled" for 335 bookings that actually canceled (false negatives). If the hotel relies on this model to decide which bookings need a retention offer, what is the practical consequence?</b></p>
<input type="radio" name="ea2" id="ea2a"><label for="ea2a"> A) The hotel wastes money sending offers to guests who would have stayed anyway</label><br>
<input type="radio" name="ea2" id="ea2b"><label for="ea2b"> B) 335 likely cancellations slip through without any intervention: empty rooms the hotel thought were filled</label><br>
<input type="radio" name="ea2" id="ea2c"><label for="ea2c"> C) The model is unusable because it has false negatives</label><br>

<p><b>Q3: The ROC curve bows well above the diagonal (AUC ≈ 0.79). If the curve hugged the diagonal instead, what would that tell you?</b></p>
<input type="radio" name="ea3" id="ea3a"><label for="ea3a"> A) The model is perfect</label><br>
<input type="radio" name="ea3" id="ea3b"><label for="ea3b"> B) The model has no ability to distinguish cancelers from non-cancelers (random guessing)</label><br>
<input type="radio" name="ea3" id="ea3c"><label for="ea3c"> C) The model has high precision but low recall</label><br>

<br>
<details><summary style="cursor:pointer; background:#E8593C; color:white; padding:8px 16px; border-radius:6px; display:inline-block;"><b>📝 Check Answers</b></summary>
<div style="margin-top:12px; padding:12px; background:#fce4ec; border-radius:8px;">
<p>✅ <b>Q1: A)</b> Low recall on the minority class is the classic imbalanced-data trade-off. The model defaults to "Not Canceled" because that class is more common.</p>
<p>✅ <b>Q2: B)</b> These are costly false negatives. You might lower the threshold (e.g. 0.4 instead of 0.5) to catch more cancellations, at the cost of more false alarms. The right threshold depends on the relative business cost of an empty room vs. a wasted retention offer.</p>
<p>✅ <b>Q3: B)</b> The diagonal = AUC 0.5 = no discrimination. The further the curve bows toward the top-left corner, the better.</p>
</div>
</details>
</div>
'''
display(HTML(quiz_html))

In [ ]:
#@title 📤 Generate Submission: Submit your best model
# Pick the best approach from the performance tracker.
# Below: weighted ensemble using OOF-optimized weights from 9.3.
# If a single model scored higher on honest CV, swap it in.

# Option A: Weighted ensemble
best_test_preds = (test_p[oof.columns[:len(w)]].values * w).sum(axis=1)

# Option B: Single best model (uncomment to use instead)
# final_single = CatBoostClassifier(**best_cb_params, random_state=42, verbose=0)
# final_single.fit(X, y)
# best_test_preds = final_single.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({'booking_id': test['booking_id'], 'is_canceled': best_test_preds})
submission.to_csv('solution.csv', index=False)
print(f"✓ Saved solution.csv ({len(submission)} rows)")
print(f"  Mean prediction: {best_test_preds.mean():.3f}")
print(f"  Std prediction:  {best_test_preds.std():.3f}")
submission.head()

In [ ]:
#@title 📊 Final Performance Tracker
print("=" * 70)
print("FULL PIPELINE PERFORMANCE TRACKER")
print("=" * 70)
summary = pd.DataFrame(performance_log)
display(summary.style.background_gradient(subset=['Score'], cmap='RdYlGn').format({'Score': '{:.4f}'}))
if len(summary) > 1:
    print(f"\nTotal: {summary['Score'].iloc[0]:.4f} → {summary['Score'].max():.4f} (+{summary['Score'].max()-summary['Score'].iloc[0]:.4f})")

In [ ]:
#@title Q11: KAGGLE PITFALLS
quiz_html = '''
<div style="border:2px solid #27ae60; border-radius:12px; padding:20px; margin:10px 0; background:#f0fff0;">
<h4>🎓 Final Reflection & Kaggle Pitfalls</h4>

<p><b>🪤 The Leaderboard Trap</b></p>
<p>Kaggle has a <b>public leaderboard</b> (scored on ~30% of test data) and a <b>private leaderboard</b> (remaining ~70%, revealed after deadline). What strategy is safer?:</p>
<input type="radio" name="ft1" id="ft1a"><label for="ft1a"> A) Submit many times, optimizing for the public score</label><br>
<input type="radio" name="ft1" id="ft1b"><label for="ft1b"> B) Trust your CV score, submit fewer times</label><br>
<br>
<details><summary style="cursor:pointer; background:#27ae60; color:white; padding:8px 16px; border-radius:6px; display:inline-block;"><b>📝 Answer</b></summary>
<div style="margin-top:12px; padding:12px; background:#e8f5e9; border-radius:8px;">
✅ <b>B) Trust your CV score.</b> If you submit 50 times tweaking for the public score, you are effectively "training" on the public test set. A complex model might rank #1 publicly but crash on the private board. <b>Prefer a stable model with good CV over a flashy public score.</b>
</div></details>

<p><b>📏 How do you know your pipeline is stable?</b></p>
<details><summary><b>Answer</b></summary>
<div style="padding:8px; background:#e8f5e9; border-radius:4px;">
Check the <b>standard deviation</b> of your CV scores. AUC = 0.82 ± 0.01 → stable, reliable. AUC = 0.82 ± 0.05 → unstable, submission could be lucky or unlucky.
</div></details>

<p><b>📊 What are the tracker scores based on?</b></p>
<details><summary><b>Answer</b></summary>
<div style="padding:8px; background:#e8f5e9; border-radius:4px;">
All scores are from <b>5-fold cross-validation on the training set</b>. The test set was used <b>only once</b>, at the end, to generate solution.csv. This prevents overfitting to test data.
</div></details>

<p><b>🤔 What would you do with 2 more hours?</b></p>
<details><summary><b>Ideas</b></summary>
<div style="padding:8px; background:#e8f5e9; border-radius:4px;">
• Run Optuna yourself (200+ trials)<br>
• More feature engineering (seasonal interactions, country×agent)<br>
• Try stacking with a GBM meta-learner + passthrough<br>
• Post-process: calibrate predicted probabilities
</div></details>

<p><b>🔁 If a single Stratified K-Fold split, by chance, gives you a suspiciously high score on one fold and a suspiciously low one on another, what should you trust?</b></p>
<details><summary><b>Answer</b></summary>
<div style="padding:8px; background:#e8f5e9; border-radius:4px;">
The average across all folds, and check the standard deviation. That's exactly why we use K-Fold instead of a single split: any one fold can be lucky or unlucky.
</div></details>

<p><b>📏 We used the same "ruler" model (GradientBoostingClassifier) for the performance tracker across the preprocessing and feature engineering steps, but a different, tuned model at the end. Why not just compare the very first score to the very last?</b></p>
<details><summary><b>Answer</b></summary>
<div style="padding:8px; background:#e8f5e9; border-radius:4px;">
Because the "ruler" steps isolate the value of DATA improvements by holding the model constant, while the final scores also credit better ALGORITHMS and TUNING. Comparing only the first score to the last would conflate both effects and overstate how much came from data work alone.
</div></details>

<p><b>📦 avg_daily_rate showed almost identical box plots for canceled vs. non-canceled bookings. Should we have dropped it?</b></p>
<details><summary><b>Answer</b></summary>
<div style="padding:8px; background:#e8f5e9; border-radius:4px;">
Not necessarily. Weak univariate signal doesn't mean zero value in combination with other features (for example inside rate_per_guest, or interacting with room_type or deposit_type). Multivariate feature selection methods, like Random Forest importance, are a better final judge than a single box plot.
</div></details>

<p><b>📉 Variance: ensembles vs. a single neural network</b></p>
<p>Averaging several tree-based models (Random Forest's bagging, or our weighted blend in Section 9) mainly attacks <b>variance</b>: each model overfits the training data a bit differently, and averaging cancels out those idiosyncratic errors. A single large neural network trained once has no such averaging built in, so run-to-run variance (from random initialization, batch order, etc.) is usually higher unless you explicitly ensemble several trained copies of it too.</p>
<details><summary><b>Answer</b></summary>
<div style="padding:8px; background:#e8f5e9; border-radius:4px;">
Both families sit on the same bias-variance trade-off, they just default to different points on it. A single decision tree or a single neural net trained once is <b>low-bias, high-variance</b>: powerful enough to fit almost anything, but sensitive to exactly which rows it saw. Bagging/ensembling many such models (Random Forest, our weighted average of 5 GBMs) keeps the low bias but drives variance down by averaging out uncorrelated mistakes, this is exactly why diverse base models (Section 9's SVM bonus) help more than near-duplicate ones. A neural network gets the same variance-reduction benefit only if you explicitly ensemble multiple trained instances (different seeds, dropout, snapshots); a single trained network doesn't get this "for free" the way bagged trees do.
</div></details>
</div>
'''
display(HTML(quiz_html))